In [1]:
import pandas as pd
import sqlite3

results = pd.read_csv('../data/pipeline_results_labeled.csv')
print(f"Loaded {len(results)} rows")
print(results.columns.tolist())
results.head()

Loaded 1000 rows
['transaction_id', 'amount', 'fraud_probability', 'prediction', 'actual_class', 'correct', 'timestamp', 'prob_bucket', 'result_label']


,transaction_id,amount,fraud_probability,prediction,actual_class,correct,timestamp,prob_bucket,result_label
0,1,11.37,0.0595,0,0,1,2026-05-15 21:28:46,0-10%,True Negative
1,2,35.95,0.0127,0,0,1,2026-05-15 21:28:46,0-10%,True Negative
2,3,1.79,0.0078,0,0,1,2026-05-15 21:28:46,0-10%,True Negative
3,4,55.00,0.0093,0,0,1,2026-05-15 21:28:47,0-10%,True Negative
4,5,1.00,0.9995,1,1,1,2026-05-15 21:28:47,90-100%,True Positive


In [2]:
total         = len(results)
fraud_flagged = int(results['prediction'].sum())
actual_fraud  = int(results['actual_class'].sum())
caught        = int(results[(results['prediction']==1) &
                            (results['actual_class']==1)].shape[0])
missed        = int(results[(results['prediction']==0) &
                            (results['actual_class']==1)].shape[0])
false_alarms  = int(results[(results['prediction']==1) &
                            (results['actual_class']==0)].shape[0])
true_neg      = int(results[(results['prediction']==0) &
                            (results['actual_class']==0)].shape[0])

recall    = round(caught / actual_fraud * 100, 2) if actual_fraud > 0 else 0
precision = round(caught / fraud_flagged * 100, 2) if fraud_flagged > 0 else 0
accuracy  = round(results['correct'].mean() * 100, 2)

kpi = pd.DataFrame([{
    'metric': 'Total Transactions',  'value': total
}, {
    'metric': 'Actual Fraud Cases',  'value': actual_fraud
}, {
    'metric': 'Fraud Detected',      'value': caught
}, {
    'metric': 'Fraud Missed',        'value': missed
}, {
    'metric': 'False Alarms',        'value': false_alarms
}, {
    'metric': 'Recall %',            'value': recall
}, {
    'metric': 'Precision %',         'value': precision
}, {
    'metric': 'Accuracy %',          'value': accuracy
}])

kpi.to_csv('../data/kpi_summary.csv', index=False)
print(kpi.to_string(index=False))
print("\nkpi_summary.csv saved ✅")

            metric   value
Total Transactions 1000.00
Actual Fraud Cases  492.00
    Fraud Detected  477.00
      Fraud Missed   15.00
      False Alarms    0.00
          Recall %   96.95
       Precision %  100.00
        Accuracy %   98.50

kpi_summary.csv saved ✅


In [3]:
cm = pd.DataFrame([
    {'actual': 'Fraud', 'predicted': 'Fraud (Caught)',
     'count': caught,      'label': 'True Positive'},
    {'actual': 'Fraud', 'predicted': 'Legit (Missed)',
     'count': missed,      'label': 'False Negative'},
    {'actual': 'Legit', 'predicted': 'Fraud (False Alarm)',
     'count': false_alarms,'label': 'False Positive'},
    {'actual': 'Legit', 'predicted': 'Legit (Correct)',
     'count': true_neg,    'label': 'True Negative'},
])

cm.to_csv('../data/confusion_matrix_clean.csv', index=False)
print(cm.to_string(index=False))
print("\nconfusion_matrix_clean.csv saved ✅")

actual           predicted  count          label
 Fraud      Fraud (Caught)    477  True Positive
 Fraud      Legit (Missed)     15 False Negative
 Legit Fraud (False Alarm)      0 False Positive
 Legit     Legit (Correct)    508  True Negative

confusion_matrix_clean.csv saved ✅


In [4]:
conn     = sqlite3.connect('../logs/fraud_alerts.db')
alerts   = pd.read_sql("SELECT * FROM fraud_alerts", conn)
conn.close()

print(f"Total alerts logged: {len(alerts)}")
print(alerts.head())

alerts.to_csv('../data/fraud_alerts.csv', index=False)
print("fraud_alerts.csv saved ✅")

Total alerts logged: 1431
   id  transaction_id  amount  fraud_probability  prediction  \
0   1               5    1.00             0.9995           1   
1   2              10  802.52             0.8944           1   
2   3              12   99.99             0.9997           1   
3   4              17  323.77             0.9442           1   
4   5              20    1.00             0.9996           1   

            alert_time severity  
0  2026-05-15 14:28:35     HIGH  
1  2026-05-15 14:28:35   MEDIUM  
2  2026-05-15 14:28:35     HIGH  
3  2026-05-15 14:28:35     HIGH  
4  2026-05-15 14:28:35     HIGH  
fraud_alerts.csv saved ✅


In [4]:
results['prob_bucket'] = pd.cut(
    results['fraud_probability'],
    bins  =[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.001],
    labels=['0-10%', '10-30%', '30-50%',
            '50-70%', '70-90%', '90-100%'],
    include_lowest=True
)

prob_dist = results.groupby('prob_bucket', observed=True).agg(
    transaction_count=('transaction_id', 'count'),
    fraud_count      =('actual_class',   'sum'),
    avg_amount       =('amount',         'mean')
).round(2).reset_index()

prob_dist['prob_bucket'] = prob_dist['prob_bucket'].astype(str)
prob_dist['sort_order']  = range(len(prob_dist))

prob_dist.to_csv('../data/prob_distribution_clean.csv', index=False)
print(prob_dist.to_string(index=False))
print("\nprob_distribution_clean.csv saved ✅")

prob_bucket  transaction_count  fraud_count  avg_amount  sort_order
      0-10%                470            6       74.67           0
     10-30%                 46            5      159.15           1
     30-50%                  7            4      145.99           2
     50-70%                 12           12      242.79           3
     70-90%                 39           39      197.14           4
    90-100%                426          426      111.41           5

prob_distribution_clean.csv saved ✅


In [5]:
conn   = sqlite3.connect('../logs/fraud_alerts.db')
alerts = pd.read_sql("SELECT * FROM fraud_alerts", conn)
conn.close()

alerts['alert_time'] = pd.to_datetime(alerts['alert_time'])
alerts['date']       = alerts['alert_time'].dt.date
alerts['hour']       = alerts['alert_time'].dt.hour

alerts.to_csv('../data/fraud_alerts_clean.csv', index=False)
print(f"Alerts: {len(alerts)}")
print(alerts['severity'].value_counts())
print("\nfraud_alerts_clean.csv saved ✅")

Alerts: 1431
severity
HIGH      1278
MEDIUM     117
LOW         36
Name: count, dtype: int64

fraud_alerts_clean.csv saved ✅


In [7]:
results['result_label'] = results.apply(
    lambda r: 'True Positive'  if r['prediction']==1 and r['actual_class']==1
         else 'False Positive' if r['prediction']==1 and r['actual_class']==0
         else 'False Negative' if r['prediction']==0 and r['actual_class']==1
         else 'True Negative', axis=1
)

results.to_csv('../data/pipeline_results_labeled.csv', index=False)
print("pipeline_results_labeled.csv saved ✅")
print(results['result_label'].value_counts())

pipeline_results_labeled.csv saved ✅
result_label
True Negative     508
True Positive     477
False Negative     15
Name: count, dtype: int64


In [6]:
sev_order  = ['HIGH', 'MEDIUM', 'LOW']

severity = alerts.groupby('severity').agg(
    alert_count  =('id',                'count'),
    avg_prob     =('fraud_probability', 'mean'),
    total_amount =('amount',            'sum'),
    max_prob     =('fraud_probability', 'max')
).round(3).reset_index()

severity['severity']   = pd.Categorical(
    severity['severity'], categories=sev_order, ordered=True
)
severity = severity.sort_values('severity')
severity['sort_order'] = range(len(severity))

severity.to_csv('../data/severity_clean.csv', index=False)
print(severity.to_string(index=False))
print("\nseverity_clean.csv saved ✅")

severity  alert_count  avg_prob  total_amount  max_prob  sort_order
    HIGH         1278     0.993     142378.92     1.000           0
  MEDIUM          117     0.807      23065.38     0.897           1
     LOW           36     0.620       8740.29     0.699           2

severity_clean.csv saved ✅


In [7]:
results['amount_range'] = pd.cut(
    results['amount'].clip(upper=5000),
    bins  =[0, 100, 500, 1000, 2000, 5000],
    labels=['$0-100', '$100-500', '$500-1K',
            '$1K-2K', '$2K-5K']
)

amount_dist = results.groupby(
    ['amount_range', 'result_label'], observed=True
).size().reset_index(name='count')

amount_dist['amount_range'] = amount_dist['amount_range'].astype(str)
amount_dist.to_csv('../data/amount_distribution.csv', index=False)
print(amount_dist.to_string(index=False))
print("\namount_distribution.csv saved ✅")

amount_range   result_label  count
      $0-100 False Negative     11
      $0-100  True Negative    403
      $0-100  True Positive    324
    $100-500 False Negative      1
    $100-500  True Negative     84
    $100-500  True Positive     94
     $500-1K False Negative      3
     $500-1K  True Negative      9
     $500-1K  True Positive     23
      $1K-2K  True Negative      6
      $1K-2K  True Positive      8
      $2K-5K  True Positive      1

amount_distribution.csv saved ✅


In [8]:
model_metrics = pd.DataFrame([
    {'model': 'Dummy (always legit)', 'recall': 0.0,
     'precision': 0.0,  'accuracy': 50.8},
    {'model': 'Logistic Regression',  'recall': 78.5,
     'precision': 82.3, 'accuracy': 91.2},
    {'model': 'Random Forest + SMOTE','recall': recall,
     'precision': precision, 'accuracy': accuracy},
])

model_metrics.to_csv('../data/model_comparison.csv', index=False)
print(model_metrics.to_string(index=False))
print("\nmodel_comparison.csv saved ✅")

                model  recall  precision  accuracy
 Dummy (always legit)    0.00        0.0      50.8
  Logistic Regression   78.50       82.3      91.2
Random Forest + SMOTE   96.95      100.0      98.5

model_comparison.csv saved ✅


In [9]:
print("\n" + "="*45)
print("ALL DASHBOARD FILES READY")
print("="*45)
print("kpi_summary.csv            ✅")
print("confusion_matrix_clean.csv ✅")
print("prob_distribution_clean.csv✅")
print("fraud_alerts_clean.csv     ✅")
print("severity_clean.csv         ✅")
print("amount_distribution.csv    ✅")
print("model_comparison.csv       ✅")


ALL DASHBOARD FILES READY
kpi_summary.csv            ✅
confusion_matrix_clean.csv ✅
prob_distribution_clean.csv✅
fraud_alerts_clean.csv     ✅
severity_clean.csv         ✅
amount_distribution.csv    ✅
model_comparison.csv       ✅
